## Middleware

In [7]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage

model = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature = 0
)

@tool
def divide(a: int, b: int) -> str:
    """Divide two numbers."""
    if b == 0:
        raise ValueError("Cannot divide by zero")

    return str(a / b)

@wrap_tool_call
def handle_tool_error(request, handler):
    try:
        return handler(request)

    except Exception as e:
        print("🔥 TOOL ERROR CAUGHT:", str(e))

        return ToolMessage(
            content=f"Tool failed: {str(e)}",
            tool_call_id=request.tool_call["id"]
        )
        
agent = create_agent(
    model=model,
    tools=[divide],
    system_prompt=(
        "You must use the divide tool for every division calculation. "
        "Never calculate division yourself."
    ),
    middleware=[handle_tool_error],
)


result = agent.invoke({"messages":[
    {
        "role" : "user",
        "content": "Divide (10,0)"
    }
]})

print(result)

🔥 TOOL ERROR CAUGHT: Cannot divide by zero
{'messages': [HumanMessage(content='Divide (10,0)', additional_kwargs={}, response_metadata={}, id='362217e6-de48-427c-86ce-352897e8fc74'), AIMessage(content='', additional_kwargs={'reasoning_content': "The user asks to divide (10,0). That's division by zero. We must use the divide tool for every division calculation. The tool likely will error or return something. We need to handle division by zero gracefully. Probably we should call the tool and see response.", 'tool_calls': [{'id': 'fc_414ca82f-5972-4d42-874c-39df4e639077', 'function': {'arguments': '{"a":10,"b":0}', 'name': 'divide'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 88, 'prompt_tokens': 145, 'total_tokens': 233, 'completion_time': 0.185689862, 'completion_tokens_details': {'reasoning_tokens': 55}, 'prompt_time': 0.006175046, 'prompt_tokens_details': None, 'queue_time': 0.369314764, 'total_time': 0.191864908}, 'model_name': 'openai/gpt-oss-120b